# Comparing Docker Build Strategies: BuildKit vs Classic vs Buildx

## Purpose

This notebook compares the three build mechanisms available in modern Docker: the legacy **classic builder**, **BuildKit** (the default build engine), and the **Buildx** plugin (which extends BuildKit with multi-platform and registry-caching capabilities). Each mechanism is demonstrated with a working example, and a feature matrix at the end summarizes when to choose each one.

## When to use

| Mechanism | Best suited for | Avoid when |
|---|---|---|
| Classic builder | Legacy Docker installations, isolated debugging of build failures on the old engine | You need parallel step execution, content-addressable caching, or build secrets |
| BuildKit | Default build workflow on modern Docker; concurrent builds with dependency graph parallelism | You need multi-platform output or registry-shared cache pools (use Buildx instead) |
| Buildx | Multi-platform image builds; sharing cache across CI runners via registry export/import; custom isolated builder instances | Single-platform, single-host builds where the default BuildKit engine already suffices |

## Prerequisites

- Docker installed and running on the host
- A directory containing a Dockerfile to build
- For multi-platform examples: binfmt_misc emulation may be required if the target architecture differs from the host

## 1. Check the current builder

Docker exposes the active builder and plugin status through 'docker info'. The Builder Version field confirms whether BuildKit is the active engine, and the Client Plugins section shows whether Buildx is installed.

In [ ]:
# last_verified: 2026-09-29 · Docker n/a

# Inspect the Docker daemon to confirm the active builder engine and available plugins
!docker info --format '{{json .}}'

## 2. Classic builder (legacy)

The classic builder is the original Docker build engine. It executes Dockerfile instructions sequentially, one layer per RUN, ADD, or COPY step, with no dependency-graph parallelism. Caching is purely positional: a change at step N invalidates every step after N. The classic engine is still reachable by setting DOCKER_BUILDKIT=0, which is useful when you need to reproduce a historical build or debug an issue that only manifests under the legacy parser.

In [ ]:
# Force the classic (legacy) builder via DOCKER_BUILDKIT=0
# --no-cache ensures we observe the full sequential build
!DOCKER_BUILDKIT=0 docker build --no-cache -t example:classic ..

## 3. BuildKit (default engine)

BuildKit is the modern build engine that is the default on modern Docker installations. It parses the Dockerfile into a dependency graph and executes independent steps in parallel, applies content-addressable caching (so moving a step earlier in the Dockerfile does not invalidate unrelated downstream cache entries), and supports inline secrets and SSH forwarding through the --secret and --ssh build flags. BuildKit is activated by setting DOCKER_BUILDKIT=1 or by relying on the daemon default.

In [ ]:
# BuildKit is activated with DOCKER_BUILDKIT=1 (default on modern Docker)
# --secret injects sensitive data at build time without baking it into the image
# --ssh forwards a local SSH agent so RUN can clone private repos
!DOCKER_BUILDKIT=1 docker build -t example:buildkit ..

## 4. Buildx (multi-platform and registry caching)

Buildx is a Docker CLI plugin that wraps BuildKit. It adds three capabilities the core engine does not expose on its own: multi-platform builds (--platform with emulation or native builders), registry-based cache import/export (--cache-from / --cache-to pointing at a registry), and the ability to instantiate dedicated, isolated builder instances with 'docker buildx create'. Buildx is the right choice when a single build must produce images for multiple architectures or when cache needs to be shared across distributed CI runners.

In [ ]:
# List available Buildx builders and their platforms
!docker buildx ls

# Build for multiple platforms from a single Dockerfile
# --load is omitted because multi-platform builds produce a manifest list in the registry
# --cache-to exports the build cache to the registry for reuse by subsequent CI runs
!docker buildx build --platform linux/amd64,linux/arm64 -t example:buildx --push ..

## Feature comparison

| Feature | Classic | BuildKit | Buildx |
|---|---|---|---|
| Sequential step execution | Yes | No (graph-parallel) | No (graph-parallel) |
| Content-addressable caching | No | Yes | Yes |
| Build secrets / SSH forwarding | No | Yes (--secret, --ssh) | Yes (--secret, --ssh) |
| Multi-platform output | No | No | Yes (--platform + --push) |
| Registry cache export/import | No | Limited | Yes (--cache-from / --cache-to) |
| Isolated builder instances | No | No | Yes (buildx create) |

## Verify

After running the examples above, confirm the results with the following checks. A successful workflow shows the three tags (example:classic, example:buildkit, example:buildx) present in the local image store, and the Buildx platform listing reports linux/amd64 and linux/arm64 availability.

In [ ]:
# Confirm all three image tags exist locally
!docker images example

# Confirm Buildx recognizes the requested platforms
!docker buildx imagetools inspect example:buildx 2>/dev/null || echo 'Run with --push to create a manifest list'

## Common errors

- 'docker: buildx is not a docker command' — the Buildx plugin is not installed. Install the docker-buildx plugin package for your platform.
- 'Cannot use docker buildx build with --load' when targeting multiple platforms — --load only works for a single platform. Use --push for multi-platform builds (the images land in the registry, not the local store).
- Classic builder ignores --secret and --ssh — these flags are BuildKit-only. Setting DOCKER_BUILDKIT=0 and passing --secret will cause the build to error.

## What I learned

- Use the classic builder only for legacy compatibility or isolated debugging — it is strictly slower due to sequential execution.
- BuildKit is the right default for everyday local and CI builds; the --mount=type=cache and --secret flags alone often halve rebuild times.
- Buildx is only necessary when you need multi-platform images or registry-shared cache pools — it adds configuration overhead that is not worth paying for single-platform builds.